In [8]:
import importlib
from pathlib import Path
import sys

# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/pratik0620/adaptive-federated-aggregation.git"

if IN_COLAB or Path("/content").exists():
    REPO_ROOT = Path("/content/adaptive-federated-aggregation")
    # If alternate clone directory exists and primary does not, use it
    if not REPO_ROOT.exists():
        alt_root = Path("/content/adaptive-federated-aggregation")
        if alt_root.exists() and (alt_root / "src").is_dir():
            REPO_ROOT = alt_root
        else:
            !git clone {REPO_URL} {REPO_ROOT}
    
    # If repository already exists, safely sync with remote and discard local dirty diffs
    if REPO_ROOT.exists() and (REPO_ROOT / ".git").is_dir():
        !git -C {REPO_ROOT} fetch origin main
        !git -C {REPO_ROOT} reset --hard origin/main
else:
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").is_dir() and (REPO_ROOT.parent / "src").is_dir():
        REPO_ROOT = REPO_ROOT.parent

# Ensure REPO_ROOT takes top precedence in sys.path
repo_path_str = str(REPO_ROOT)
if repo_path_str in sys.path:
    sys.path.remove(repo_path_str)
sys.path.insert(0, repo_path_str)

# Invalidate import caches and clear cached src modules to prevent stale imports
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

SRC_DIR = REPO_ROOT / "src"
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f"src directory not found at {SRC_DIR}")

print("Repository:", REPO_ROOT)
print("Source modules:", SRC_DIR)

Repository: e:\College\Third Year\Data Science\Course Project\Project Repo
Source modules: e:\College\Third Year\Data Science\Course Project\Project Repo\src


In [2]:
# Install the packages required by this notebook and the shared preprocessing utilities.
%pip install -q numpy pandas scikit-learn matplotlib jupyter


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Admin\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


## Configuration — Google Drive Paths

The next code cell defines the only machine-specific setting, `BASE_DRIVE_PATH`, and validates the three NIH metadata files. This notebook does not require an image directory and does not download or copy X-ray images.

# NIH ChestX-ray14 preprocessing

This notebook creates the NIH federated client as metadata-only CSV files. It keeps clean target positives and explicit `No Finding` negatives, excludes ambiguous label combinations, preserves patient IDs, and splits whole patients. Images are never copied or downloaded in this phase.

**Storage model:** Code and notebooks are stored in GitHub, while the large NIH metadata files and future image files are stored in Google Drive. The actual X-ray images will be downloaded later, after the target disease and required sample size are finalized.

In [9]:
import json, platform
import numpy as np
import pandas as pd

from src.config import TARGET_DISEASE, RANDOM_SEED, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES
from src.dataset_utils import nih_clean_binary_labels, standardize_nih
from src.preprocessing_utils import (
    assert_no_leakage,
    assert_no_study_leakage,
    build_summary,
    require_file,
    sample_clean_cases,
    save_splits,
    save_summary,
    set_reproducibility,
    split_train_val_by_patient,
)

set_reproducibility(RANDOM_SEED)
print('Python:', platform.python_version())
print('pandas:', pd.__version__, 'numpy:', np.__version__)

# Google Drive configuration for raw NIH metadata and metadata-only outputs.
BASE_DRIVE_PATH = Path("/content/drive/MyDrive/Healthcare_FL")
# Local fallback if running outside Google Colab
if not BASE_DRIVE_PATH.exists() and (REPO_ROOT / "data").exists():
    BASE_DRIVE_PATH = REPO_ROOT

NIH_ROOT = BASE_DRIVE_PATH / "data" / "raw" / "NIH" if (BASE_DRIVE_PATH / "data" / "raw" / "NIH").exists() else BASE_DRIVE_PATH / "raw" / "NIH"
NIH_METADATA_DIR = NIH_ROOT / "metadata" if (NIH_ROOT / "metadata").exists() else NIH_ROOT

METADATA_FILE = NIH_METADATA_DIR / "Data_Entry_2017.csv"
TRAIN_VAL_FILE = NIH_METADATA_DIR / "train_val_list.txt"
TEST_FILE = NIH_METADATA_DIR / "test_list.txt"
OUTPUT_DIR = BASE_DRIVE_PATH / "data" / "processed" / "NIH" if (BASE_DRIVE_PATH / "data").exists() else BASE_DRIVE_PATH / "processed" / "NIH"

# Project-level disease names stay stable; each source can map them to its own label vocabulary.
NIH_LABEL_MAP = {"Pleural Effusion": "Effusion"}
NIH_TARGET_LABEL = NIH_LABEL_MAP.get(TARGET_DISEASE, TARGET_DISEASE)

metadata_path = require_file(METADATA_FILE, "NIH metadata CSV")
train_val_path = require_file(TRAIN_VAL_FILE, "NIH train/validation split list")
test_path = require_file(TEST_FILE, "NIH test split list")

print("Repository:", REPO_ROOT)
print("Data Path:", BASE_DRIVE_PATH)
print("Dataset:", NIH_ROOT)
print("Metadata:", METADATA_FILE)
print("NIH train/validation split path:", TRAIN_VAL_FILE)
print("NIH test split path:", TEST_FILE)
print("Project target disease:", TARGET_DISEASE)
print("NIH source target label:", NIH_TARGET_LABEL)
print("Output Directory:", OUTPUT_DIR)
print("All required NIH metadata files exist.")

Python: 3.13.9
pandas: 2.3.3 numpy: 2.2.6
Repository: e:\College\Third Year\Data Science\Course Project\Project Repo
Data Path: e:\College\Third Year\Data Science\Course Project\Project Repo
Dataset: e:\College\Third Year\Data Science\Course Project\Project Repo\data\raw\NIH
Metadata: e:\College\Third Year\Data Science\Course Project\Project Repo\data\raw\NIH\metadata\Data_Entry_2017.csv
NIH train/validation split path: e:\College\Third Year\Data Science\Course Project\Project Repo\data\raw\NIH\metadata\train_val_list.txt
NIH test split path: e:\College\Third Year\Data Science\Course Project\Project Repo\data\raw\NIH\metadata\test_list.txt
Project target disease: Pleural Effusion
NIH source target label: Effusion
Output Directory: e:\College\Third Year\Data Science\Course Project\Project Repo\data\processed\NIH
All required NIH metadata files exist.


## Load and inspect
NIH `Finding Labels` is pipe-separated. The clean binary policy is deliberately conservative: a target-containing study is positive; exactly `No Finding` is negative; every other combination is excluded.

In [10]:
nih = pd.read_csv(metadata_path)
train_val_ids = pd.read_csv(train_val_path, header=None, names=['image_id'])
test_ids = pd.read_csv(test_path, header=None, names=['image_id'])
train_val_ids['image_id'] = train_val_ids['image_id'].astype(str).str.strip()
test_ids['image_id'] = test_ids['image_id'].astype(str).str.strip()

print('images:', len(nih))
print('unique patients:', nih['Patient ID'].nunique())
print('columns:', nih.columns.tolist())
print('official train/validation image IDs:', len(train_val_ids))
print('official test image IDs:', len(test_ids))
print('official split image overlap:', len(set(train_val_ids['image_id']) & set(test_ids['image_id'])))
display(nih.isna().sum().sort_values(ascending=False).head(20).to_frame('missing'))
display(nih['Finding Labels'].value_counts(dropna=False).head(20).to_frame('label_count'))

# NIH ChestX-ray14 uses 'Effusion' for 'Pleural Effusion'.
# Positive: Finding Labels contains 'Effusion' (including multi-label rows)
# Negative: Finding Labels is exactly 'No Finding'
# All other label combinations are excluded.
clean, counts = nih_clean_binary_labels(
    nih,
    target_disease=TARGET_DISEASE,
    source_target_label=NIH_TARGET_LABEL,
)
print('Counts before filtering:', counts)
print('Clean rows:', len(clean))

images: 112120
unique patients: 30805
columns: ['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11']
official train/validation image IDs: 86524
official test image IDs: 25596
official split image overlap: 0


,missing
Unnamed: 11,112120
Image Index,0
Follow-up #,0
Finding Labels,0
Patient ID,0
Patient Age,0
View Position,0
Patient Gender,0
OriginalImage[Width,0
Height],0


,label_count
Finding Labels,
No Finding,60361
Infiltration,9547
Atelectasis,4215
Effusion,3955
Nodule,2705
Pneumothorax,2194
Mass,2139
Effusion|Infiltration,1603
Atelectasis|Infiltration,1350


Counts before filtering: {'target_disease': 'Pleural Effusion', 'source_target_label': 'Effusion', 'positive_before_filter': 13317, 'negative_before_filter': 60361, 'excluded_ambiguous_before_filter': 38442}
Clean rows: 73678


In [11]:
# No image directory is required for this metadata-only phase.
# The configured future image location is only stored as a path reference; no file is opened or checked.
standardized = standardize_nih(clean, image_dir=NIH_ROOT / 'images')
standardized['image_id'] = standardized['image_id'].astype(str).str.strip()

official_train_val_image_ids = set(train_val_ids['image_id'])
official_test_image_ids = set(test_ids['image_id'])
if official_train_val_image_ids & official_test_image_ids:
    raise AssertionError('Official NIH train/validation and test lists overlap.')

official_train_val = standardized[standardized['image_id'].isin(official_train_val_image_ids)].copy()
official_test = standardized[standardized['image_id'].isin(official_test_image_ids)].copy()
missing_from_official_lists = set(standardized['image_id']) - (official_train_val_image_ids | official_test_image_ids)
print('Clean images missing from official lists:', len(missing_from_official_lists))

# Sampling is applied only to the official train/validation portion so the official test set is preserved.
official_train_val = sample_clean_cases(
    official_train_val,
    MAX_POSITIVE_SAMPLES,
    MAX_NEGATIVE_SAMPLES,
    RANDOM_SEED,
)

# Patient-level split of the official train/val portion into train and validation sets.
# Preserves the official NIH test set as the test split.
train_val_splits = split_train_val_by_patient(official_train_val, seed=RANDOM_SEED, val_size=0.1)
splits = {
    'train': train_val_splits['train'],
    'val': train_val_splits['val'],
    'test': official_test,
}

assert_no_leakage(splits)
assert_no_study_leakage(splits)
print('patient leakage checks: passed')
print('study leakage checks: passed where study_id exists')

print('positive count:', counts['positive_before_filter'])
print('negative count:', counts['negative_before_filter'])
print('excluded count:', counts['excluded_ambiguous_before_filter'])
print('official split usage status: True')
for split_name, split_frame in splits.items():
    print(
        f'{split_name} image count:', len(split_frame),
        '| positive:', int((split_frame['target'] == 1).sum()),
        '| negative:', int((split_frame['target'] == 0).sum()),
        '| unique patients:', split_frame['patient_id'].nunique(),
    )

display(pd.DataFrame([
    {
        'split': split_name,
        'images': len(split_frame),
        'positive': int((split_frame['target'] == 1).sum()),
        'negative': int((split_frame['target'] == 0).sum()),
        'unique_patients': split_frame['patient_id'].nunique(),
    }
    for split_name, split_frame in splits.items()
]))
display(splits['train'].head())

save_splits(splits, OUTPUT_DIR)
summary = build_summary(
    splits,
    'NIH',
    {
        'target_disease': TARGET_DISEASE,
        'source_target_label': NIH_TARGET_LABEL,
        'random_seed': RANDOM_SEED,
        'max_positive_samples_train_val': MAX_POSITIVE_SAMPLES,
        'max_negative_samples_train_val': MAX_NEGATIVE_SAMPLES,
        'filter_counts_before_filtering': counts,
        'official_split_lists_loaded': True,
        'official_split_used': True,
        'official_test_preserved': True,
        'image_paths_materialized': False,
    },
)
save_summary(summary, OUTPUT_DIR)
print(json.dumps(summary, indent=2))
print('Saved:', sorted(path.name for path in OUTPUT_DIR.glob('*')))

Clean images missing from official lists: 0
patient leakage checks: passed
study leakage checks: passed where study_id exists
positive count: 13317
negative count: 60361
excluded count: 38442
official split usage status: True
train image count: 53323 | positive: 7842 | negative: 45481 | unique patients: 20977
val image count: 5836 | positive: 817 | negative: 5019 | unique patients: 2331
test image count: 14519 | positive: 4658 | negative: 9861 | unique patients: 2615


,split,images,positive,negative,unique_patients
0,train,53323,7842,45481,20977
1,val,5836,817,5019,2331
2,test,14519,4658,9861,2615


,image_id,patient_id,study_id,source,target,age,sex,view_position,projection,image_width,image_height,pixel_spacing_x,pixel_spacing_y,scanner_manufacturer,image_path,finding_labels,follow_up
0,00013499_000.png,13499,NaN,NIH,1,33,M,PA,PA,NaN,NaN,NaN,NaN,NaN,e:\College\Third Year\Data Science\Course Proj...,Effusion|Mass|Nodule,0
1,00025157_000.png,25157,NaN,NIH,0,30,F,PA,PA,NaN,NaN,NaN,NaN,NaN,e:\College\Third Year\Data Science\Course Proj...,No Finding,0
2,00006040_005.png,6040,NaN,NIH,0,18,F,PA,PA,NaN,NaN,NaN,NaN,NaN,e:\College\Third Year\Data Science\Course Proj...,No Finding,5
3,00024290_000.png,24290,NaN,NIH,0,29,M,PA,PA,NaN,NaN,NaN,NaN,NaN,e:\College\Third Year\Data Science\Course Proj...,No Finding,0
4,00004137_000.png,4137,NaN,NIH,0,40,F,PA,PA,NaN,NaN,NaN,NaN,NaN,e:\College\Third Year\Data Science\Course Proj...,No Finding,0


{
  "source": "NIH",
  "parameters": {
    "target_disease": "Pleural Effusion",
    "source_target_label": "Effusion",
    "random_seed": 42,
    "max_positive_samples_train_val": null,
    "max_negative_samples_train_val": null,
    "filter_counts_before_filtering": {
      "target_disease": "Pleural Effusion",
      "source_target_label": "Effusion",
      "positive_before_filter": 13317,
      "negative_before_filter": 60361,
      "excluded_ambiguous_before_filter": 38442
    },
    "official_split_lists_loaded": true,
    "official_split_used": true,
    "official_test_preserved": true,
    "image_paths_materialized": false
  },
  "splits": {
    "train": {
      "images": 53323,
      "patients": 20977,
      "studies": 0,
      "positive": 7842,
      "negative": 45481
    },
    "val": {
      "images": 5836,
      "patients": 2331,
      "studies": 0,
      "positive": 817,
      "negative": 5019
    },
    "test": {
      "images": 14519,
      "patients": 2615,
      "studi